In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import joblib

In [3]:
df = pd.read_csv("data/insurance.csv")
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print(df.head(3))

(1338, 7)
age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64
   age     sex    bmi  children smoker     region     charges
0   19  female  27.90         0    yes  southwest  16884.9240
1   18    male  33.77         1     no  southeast   1725.5523
2   28    male  33.00         3     no  southeast   4449.4620


In [4]:
le = LabelEncoder()
text_cols = ['sex', 'smoker', 'region']

for col in text_cols:
    df[col] = le.fit_transform(df[col])

print(df.dtypes)
print("\nCharges skewness:", df['charges'].skew().round(2))
print(df['charges'].describe())

age           int64
sex           int64
bmi         float64
children      int64
smoker        int64
region        int64
charges     float64
dtype: object

Charges skewness: 1.52
count     1338.000000
mean     13270.422265
std      12110.011237
min       1121.873900
25%       4740.287150
50%       9382.033000
75%      16639.912515
max      63770.428010
Name: charges, dtype: float64


In [5]:
df['log_charges'] = np.log1p(df['charges'])
print("Skewness after log transform:", df['log_charges'].skew().round(2))

Skewness after log transform: -0.09


In [6]:
X = df.drop(columns=['charges', 'log_charges'])
y = df['log_charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Training done!")

Training done!


In [7]:
y_pred_log = model.predict(X_test)
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

r2 = r2_score(y_test, y_pred_log)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : ${rmse:,.2f}")

R² Score : 0.8503
RMSE     : $4,389.68


In [8]:
# Save encoders for each text column
df_orig = pd.read_csv("data/insurance.csv")
encoders = {}
text_cols = ['sex', 'smoker', 'region']

for col in text_cols:
    le_col = LabelEncoder()
    le_col.fit(df_orig[col])
    encoders[col] = le_col

joblib.dump(model, 'model/insurance_model.pkl')
joblib.dump(encoders, 'model/encoders.pkl')
print("Saved!")
print("Sex values:", list(encoders['sex'].classes_))
print("Smoker values:", list(encoders['smoker'].classes_))
print("Region values:", list(encoders['region'].classes_))

Saved!
Sex values: ['female', 'male']
Smoker values: ['no', 'yes']
Region values: ['northeast', 'northwest', 'southeast', 'southwest']
